# Unsupervised Retinal Vessel Segmentation
## Self-Supervised Attention U-Net with ASPP + Two-Phase Training

**Architecture:** Modified Attention U-Net + Atrous Spatial Pyramid Pooling (ASPP)  
**Learning:** Fully unsupervised — no manual annotations required  
**Phase 1:** Frangi + Sato vesselness pseudo-label training  
**Phase 2:** Self-training with high-confidence model predictions  
**Dataset:** FD3611 (Diabetic Retinopathy, Media Hazy, Myopic Retinopathy, Normal, Optic Disc Disorder)  
**Target:** Google Colab T4 GPU (15 GB VRAM)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 ─ Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q scikit-image opencv-python-headless albumentations tqdm
print('All packages ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 ─ Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 ─ Imports
# ─────────────────────────────────────────────────────────────────────────────
import os, glob, random, warnings, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm

import cv2
from scipy import ndimage
from skimage.filters import frangi, sato, threshold_otsu
from skimage.morphology import (
    skeletonize, remove_small_objects, disk, binary_dilation, binary_erosion
)
from skimage.measure import label, regionprops

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torch.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split

# ── Reproducibility ──
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 ─ Configuration
# ─────────────────────────────────────────────────────────────────────────────
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    # Update DATA_ROOT to the actual location of your FD3611 folder on Drive
    DATA_ROOT   = '/content/drive/MyDrive/FD3611'
    OUTPUT_DIR  = '/content/vessel_output'

    # ── Dataset ────────────────────────────────────────────────────────────
    CATEGORIES  = [
        'Diabetic_Retinopathy',
        'Media_Hazy',
        'Myopic_Retinopathy',
        'Normal',
        'Optic_Disc_Disorder',
    ]
    IMG_SIZE    = 512          # resize all images to this

    # ── Training ───────────────────────────────────────────────────────────
    BATCH_SIZE         = 6     # safe for T4 15 GB at 512×512
    NUM_EPOCHS_PHASE1  = 30    # Frangi pseudo-label training
    NUM_EPOCHS_PHASE2  = 20    # self-training refinement
    LR                 = 1e-4
    WEIGHT_DECAY       = 1e-5
    USE_AMP            = True  # mixed precision

    # ── Model ──────────────────────────────────────────────────────────────
    IN_CHANNELS  = 3
    OUT_CHANNELS = 1
    FEATURES     = [64, 128, 256, 512]
    DROPOUT      = 0.2

    # ── Pseudo-labels ──────────────────────────────────────────────────────
    FRANGI_SIGMAS        = range(1, 6)
    CONFIDENCE_THRESHOLD = 0.78

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print('Config ready. Output dir:', cfg.OUTPUT_DIR)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 ─ Preprocessing helpers
# ─────────────────────────────────────────────────────────────────────────────
def preprocess_retinal(img_path: str, size: int = 512):
    """
    Returns
    -------
    img_rgb   : (H,W,3) uint8  — original resized
    img_clahe : (H,W,3) uint8  — CLAHE-enhanced RGB
    green_cl  : (H,W)   uint8  — CLAHE green channel (best vessel contrast)
    """
    bgr = cv2.imread(img_path)
    if bgr is None:
        raise FileNotFoundError(f'Cannot read: {img_path}')
    img_rgb = cv2.cvtColor(
        cv2.resize(bgr, (size, size), interpolation=cv2.INTER_LANCZOS4),
        cv2.COLOR_BGR2RGB
    )
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    channels = [clahe.apply(img_rgb[:, :, c]) for c in range(3)]
    img_clahe = np.stack(channels, axis=2)
    green_cl  = channels[1]          # green channel with CLAHE
    return img_rgb, img_clahe, green_cl


def make_pseudo_label(green_ch, sigmas=range(1, 6)):
    """
    Ensemble Frangi + Sato vesselness → Otsu threshold → morphological cleanup.

    Returns
    -------
    prob   : (H,W) float32  soft probability map [0,1]
    binary : (H,W) float32  binary mask {0,1}
    """
    g = green_ch.astype(np.float64) / 255.0

    # Frangi
    fmap = frangi(g, sigmas=sigmas, alpha=0.5, beta=0.5, gamma=15,
                  black_ridges=False)
    if fmap.max() > 0:
        fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min())

    # Sato
    smap = sato(g, sigmas=sigmas, black_ridges=False)
    if smap.max() > 0:
        smap = (smap - smap.min()) / (smap.max() - smap.min())

    # Weighted ensemble
    prob = (0.60 * fmap + 0.40 * smap).astype(np.float32)

    # Otsu threshold
    thr    = threshold_otsu(prob)
    binary = prob > thr

    # Morphological cleanup
    binary = remove_small_objects(binary, min_size=25)
    binary = binary_dilation(binary, disk(1))
    binary = ndimage.binary_fill_holes(binary)

    return prob, binary.astype(np.float32)


print('Preprocessing helpers defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 ─ Scan dataset
# ─────────────────────────────────────────────────────────────────────────────
def scan_dataset(root: str, categories: list) -> pd.DataFrame:
    records = []
    for cat in categories:
        cat_dir = os.path.join(root, cat)
        if not os.path.isdir(cat_dir):
            print(f'  [WARN] Folder not found: {cat_dir}')
            continue
        pngs = glob.glob(os.path.join(cat_dir, '**', '*.png'), recursive=True)
        pngs += glob.glob(os.path.join(cat_dir, '*.PNG'), recursive=True)
        pngs = list(set(pngs))
        for p in pngs:
            records.append({'path': p, 'category': cat})
        print(f'  {cat:<30s}: {len(pngs):>5d} images')
    df = pd.DataFrame(records)
    print(f'\n  Total: {len(df)} images')
    return df

print('Scanning dataset …')
df_all = scan_dataset(cfg.DATA_ROOT, cfg.CATEGORIES)
df_all.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 ─ Generate Frangi pseudo-labels for every image  (cached to .npy)
# ─────────────────────────────────────────────────────────────────────────────
PSEUDO_DIR = os.path.join(cfg.OUTPUT_DIR, 'pseudo_labels')
os.makedirs(PSEUDO_DIR, exist_ok=True)

pseudo_paths, errors = [], []

for i, row in tqdm(df_all.iterrows(), total=len(df_all),
                   desc='Generating Frangi pseudo-labels'):
    stem    = os.path.splitext(os.path.basename(row['path']))[0]
    out_p   = os.path.join(PSEUDO_DIR, f'{stem}_{i}.npy')

    if os.path.exists(out_p):
        pseudo_paths.append(out_p)
        continue

    try:
        _, img_cl, g_cl = preprocess_retinal(row['path'], cfg.IMG_SIZE)
        prob, _         = make_pseudo_label(g_cl, cfg.FRANGI_SIGMAS)
        np.save(out_p, prob)
        pseudo_paths.append(out_p)
    except Exception as e:
        errors.append((row['path'], str(e)))
        pseudo_paths.append(None)

df_all['pseudo_path'] = pseudo_paths
df_all = df_all.dropna(subset=['pseudo_path']).reset_index(drop=True)
print(f'Pseudo-labels ready: {len(df_all)}  |  Errors: {len(errors)}')
if errors:
    print('Failed files:', [e[0] for e in errors[:5]])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 ─ Dataset & DataLoader
# ─────────────────────────────────────────────────────────────────────────────
class RetinalDataset(Dataset):
    _MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(self, df: pd.DataFrame, cfg, augment: bool = True):
        self.df      = df.reset_index(drop=True)
        self.cfg     = cfg
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        _, img_cl, _ = preprocess_retinal(row['path'], self.cfg.IMG_SIZE)
        pseudo        = np.load(row['pseudo_path']).astype(np.float32)

        # HWC → CHW, normalise to [0,1]
        img_t    = torch.from_numpy(img_cl.transpose(2, 0, 1)).float().div(255.0)
        pseudo_t = torch.from_numpy(pseudo).unsqueeze(0)          # [1,H,W]

        if self.augment:
            img_t, pseudo_t = self._augment(img_t, pseudo_t)

        img_norm = (img_t - self._MEAN) / self._STD

        return {
            'image'    : img_norm,       # normalised input for the model
            'img_raw'  : img_t,          # unnormalised for reconstruction loss
            'pseudo'   : pseudo_t,       # pseudo-label [0,1]
            'category' : row['category'],
            'path'     : row['path'],
        }

    @staticmethod
    def _augment(img, lbl):
        if random.random() > 0.5:
            img, lbl = TF.hflip(img), TF.hflip(lbl)
        if random.random() > 0.5:
            img, lbl = TF.vflip(img), TF.vflip(lbl)
        if random.random() > 0.5:
            angle = random.uniform(-30, 30)
            img   = TF.rotate(img, angle)
            lbl   = TF.rotate(lbl, angle)
        if random.random() > 0.4:
            b = random.uniform(0.8, 1.2)
            img = torch.clamp(img * b, 0, 1)
        return img, lbl


train_df, val_df = train_test_split(
    df_all, test_size=0.15, random_state=42, stratify=df_all['category']
)

train_ds = RetinalDataset(train_df, cfg, augment=True)
val_ds   = RetinalDataset(val_df,   cfg, augment=False)

LOADER_KWARGS = dict(num_workers=2, pin_memory=True)
train_loader  = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                           shuffle=True, drop_last=True, **LOADER_KWARGS)
val_loader    = DataLoader(val_ds,   batch_size=4,
                           shuffle=False, **LOADER_KWARGS)

print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 ─ Model Architecture
#
#  Self-Supervised Attention-UNet + ASPP Bottleneck
#  ┌─────────────────────────────────────────────────┐
#  │  Encoder (4 stages, MaxPool downsampling)        │
#  │  ASPP Bottleneck (multi-scale dilated conv)      │
#  │  Decoder (4 stages, bilinear up + AttnGate skip) │
#  │  Head 1 → vessel probability map  [0,1]          │
#  │  Head 2 → image reconstruction   [0,1]           │
#  └─────────────────────────────────────────────────┘
# ─────────────────────────────────────────────────────────────────────────────

class ConvBNReLU(nn.Module):
    def __init__(self, ic, oc, k=3, p=1, d=1, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(ic, oc, k, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DoubleConv(nn.Module):
    def __init__(self, ic, oc, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNReLU(ic, oc, dropout=dropout),
            ConvBNReLU(oc, oc, dropout=dropout),
        )

    def forward(self, x):
        return self.net(x)


class AttentionGate(nn.Module):
    """Soft spatial attention gate (Oktay et al., 2018)."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.Wg  = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True),
                                  nn.BatchNorm2d(F_int))
        self.Wx  = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True),
                                  nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True),
                                  nn.BatchNorm2d(1),
                                  nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        """g = gating signal from decoder; x = skip from encoder."""
        return x * self.psi(self.relu(self.Wg(g) + self.Wx(x)))


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling for multi-scale vessel context."""
    def __init__(self, ic, oc):
        super().__init__()
        self.c1  = ConvBNReLU(ic, oc, k=1, p=0)
        self.c6  = ConvBNReLU(ic, oc, k=3, p=6,  d=6)
        self.c12 = ConvBNReLU(ic, oc, k=3, p=12, d=12)
        self.c18 = ConvBNReLU(ic, oc, k=3, p=18, d=18)
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ic, oc, 1, bias=False),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
        )
        self.proj = nn.Sequential(
            nn.Conv2d(oc * 5, oc, 1, bias=False),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        gap  = F.interpolate(self.gap(x), size=(h, w),
                             mode='bilinear', align_corners=False)
        return self.proj(torch.cat(
            [self.c1(x), self.c6(x), self.c12(x), self.c18(x), gap], dim=1
        ))


class VesselNet(nn.Module):
    """
    Self-Supervised Attention U-Net with ASPP bottleneck.
    Dual output heads:
      vessel_map  – vessel probability [0,1]
      recon       – image reconstruction [0,1]
    """
    def __init__(self, in_ch=3, out_ch=1,
                 features=(64, 128, 256, 512), dropout=0.2):
        super().__init__()
        f = list(features)

        # ── Encoder ──────────────────────────────────────────────────────
        self.enc1 = DoubleConv(in_ch,  f[0], dropout)
        self.enc2 = DoubleConv(f[0],   f[1], dropout)
        self.enc3 = DoubleConv(f[1],   f[2], dropout)
        self.enc4 = DoubleConv(f[2],   f[3], dropout)
        self.pool = nn.MaxPool2d(2)

        # ── Bottleneck ────────────────────────────────────────────────────
        self.bottleneck = ASPP(f[3], f[3])

        # ── Attention gates ───────────────────────────────────────────────
        self.att4 = AttentionGate(f[3], f[3], f[2])
        self.att3 = AttentionGate(f[2], f[2], f[1])
        self.att2 = AttentionGate(f[1], f[1], f[0])
        self.att1 = AttentionGate(f[0], f[0], f[0] // 2)

        # ── Decoder ───────────────────────────────────────────────────────
        self.up4  = nn.ConvTranspose2d(f[3], f[3], 2, 2)
        self.dec4 = DoubleConv(f[3] * 2, f[2], dropout)

        self.up3  = nn.ConvTranspose2d(f[2], f[2], 2, 2)
        self.dec3 = DoubleConv(f[2] * 2, f[1], dropout)

        self.up2  = nn.ConvTranspose2d(f[1], f[1], 2, 2)
        self.dec2 = DoubleConv(f[1] * 2, f[0], dropout)

        self.up1  = nn.ConvTranspose2d(f[0], f[0], 2, 2)
        self.dec1 = DoubleConv(f[0] * 2, f[0], dropout)

        # ── Output heads ─────────────────────────────────────────────────
        self.vessel_head = nn.Sequential(
            nn.Conv2d(f[0], f[0] // 2, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(f[0] // 2, out_ch, 1), nn.Sigmoid(),
        )
        self.recon_head = nn.Sequential(
            nn.Conv2d(f[0], f[0] // 2, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(f[0] // 2, in_ch, 1), nn.Sigmoid(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b  = self.bottleneck(self.pool(e4))

        # Decoder with attention-gated skip connections
        d4 = self.dec4(torch.cat([self.up4(b),  self.att4(self.up4(b),  e4)], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), self.att3(self.up3(d4), e3)], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), self.att2(self.up2(d3), e2)], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), self.att1(self.up1(d2), e1)], 1))

        return self.vessel_head(d1), self.recon_head(d1)


# ── Sanity check ────────────────────────────────────────────────────────────
model = VesselNet(cfg.IN_CHANNELS, cfg.OUT_CHANNELS,
                  cfg.FEATURES, cfg.DROPOUT).to(device)

n_total  = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters total    : {n_total:,}')
print(f'Parameters trainable: {n_train:,}')

with torch.no_grad():
    _x = torch.randn(2, 3, 512, 512, device=device)
    _v, _r = model(_x)
    print(f'Vessel head output  : {tuple(_v.shape)}')
    print(f'Recon  head output  : {tuple(_r.shape)}')
del _x, _v, _r
torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 ─ Loss Functions
# ─────────────────────────────────────────────────────────────────────────────

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        p, t   = pred.reshape(-1), target.reshape(-1)
        inter  = (p * t).sum()
        return 1.0 - (2.0 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)


class WeightedBCEDice(nn.Module):
    """Weighted BCE + Dice to handle class imbalance (vessels are rare)."""
    def __init__(self, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce_w  = bce_w
        self.dice_w = dice_w
        self.dice   = DiceLoss()

    def forward(self, pred, target):
        pos_r  = target.mean().clamp(1e-4, 1 - 1e-4)
        pw     = torch.clamp((1 - pos_r) / pos_r, 1.0, 10.0)
        bce_px = F.binary_cross_entropy(pred, target, reduction='none')
        wmap   = torch.where(target > 0.5,
                             pw * torch.ones_like(target),
                             torch.ones_like(target))
        bce = (bce_px * wmap).mean()
        return self.bce_w * bce + self.dice_w * self.dice(pred, target)


class ContrastiveMarginLoss(nn.Module):
    """Pulls vessel predictions up, background predictions down."""
    def __init__(self, margin: float = 0.20):
        super().__init__()
        self.margin = margin

    def forward(self, pred, pseudo):
        B    = pred.shape[0]
        v    = pred.view(B, -1)
        p    = pseudo.view(B, -1)
        pmsk = (p > 0.5).float()
        nmsk = (p < 0.1).float()
        pos  = (pmsk * v).sum(1) / (pmsk.sum(1) + 1e-6)
        neg  = (nmsk * v).sum(1) / (nmsk.sum(1) + 1e-6)
        return F.relu(neg - pos + self.margin).mean()


class TotalLoss(nn.Module):
    def __init__(self, alpha=0.40, beta=0.40, gamma=0.20):
        super().__init__()
        self.alpha      = alpha    # vessel loss
        self.beta       = beta     # reconstruction loss
        self.gamma      = gamma    # contrastive loss
        self.vessel_fn  = WeightedBCEDice()
        self.recon_fn   = nn.MSELoss()
        self.contrast   = ContrastiveMarginLoss()

    def forward(self, v_pred, r_pred, v_target, r_target):
        lv = self.vessel_fn(v_pred, v_target)
        lr = self.recon_fn(r_pred, r_target)
        lc = self.contrast(v_pred, v_target)
        lt = self.alpha * lv + self.beta * lr + self.gamma * lc
        return {'total': lt, 'vessel': lv, 'recon': lr, 'contrast': lc}


print('Loss functions defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 ─ Train / Validate helpers
# ─────────────────────────────────────────────────────────────────────────────

def run_epoch(model, loader, criterion, optimizer, scaler, device, cfg,
              train: bool = True, desc: str = ''):
    model.train() if train else model.eval()
    sums = {'total': 0.0, 'vessel': 0.0, 'recon': 0.0, 'contrast': 0.0}

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc=desc, leave=False):
            imgs    = batch['image'].to(device)
            raw     = batch['img_raw'].to(device)
            pseudo  = batch['pseudo'].to(device)

            if train:
                optimizer.zero_grad()

            with autocast('cuda', enabled=cfg.USE_AMP):
                v_pred, r_pred = model(imgs)
                losses         = criterion(v_pred, r_pred, pseudo, raw)

            if train:
                scaler.scale(losses['total']).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            for k in sums:
                sums[k] += losses[k].item()

    n = len(loader)
    return {k: v / n for k, v in sums.items()}


print('Training helpers defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 ─ Phase 1: Train with Frangi pseudo-labels
# ─────────────────────────────────────────────────────────────────────────────
model      = VesselNet(cfg.IN_CHANNELS, cfg.OUT_CHANNELS,
                       cfg.FEATURES, cfg.DROPOUT).to(device)
optimizer  = torch.optim.AdamW(model.parameters(),
                                lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                 optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion  = TotalLoss(alpha=0.40, beta=0.40, gamma=0.20)
scaler     = GradScaler('cuda', enabled=cfg.USE_AMP)

CKPT_PATH  = os.path.join(cfg.OUTPUT_DIR, 'best_model.pth')
history    = {'train': [], 'val': []}
best_loss  = float('inf')

print('=' * 60)
print('PHASE 1 — Frangi pseudo-label training')
print('=' * 60)

for epoch in range(cfg.NUM_EPOCHS_PHASE1):
    tr = run_epoch(model, train_loader, criterion, optimizer, scaler,
                   device, cfg, train=True,
                   desc=f'P1 Epoch {epoch+1}/{cfg.NUM_EPOCHS_PHASE1} [train]')
    vl = run_epoch(model, val_loader,   criterion, optimizer, scaler,
                   device, cfg, train=False,
                   desc=f'P1 Epoch {epoch+1}/{cfg.NUM_EPOCHS_PHASE1} [val]')
    scheduler.step()

    history['train'].append(tr)
    history['val'].append(vl)

    tag = ''
    if vl['total'] < best_loss:
        best_loss = vl['total']
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_loss': best_loss}, CKPT_PATH)
        tag = '  ← best'

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Ep {epoch+1:03d}  '
              f'train={tr["total"]:.4f}  '
              f'val={vl["total"]:.4f}  '
              f'vessel={vl["vessel"]:.4f}{tag}')

print(f'\nPhase 1 done. Best val loss: {best_loss:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 ─ Phase 2: Self-training with refined pseudo-labels
# ─────────────────────────────────────────────────────────────────────────────

# ── Load best Phase-1 checkpoint ─────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH)['model_state_dict'])
model.eval()

# ── Generate model predictions on the full dataset ───────────────────────────
full_ds     = RetinalDataset(df_all, cfg, augment=False)
full_loader = DataLoader(full_ds, batch_size=4, shuffle=False, **LOADER_KWARGS)

REFINED_DIR = os.path.join(cfg.OUTPUT_DIR, 'refined_labels')
os.makedirs(REFINED_DIR, exist_ok=True)

refined_map = {}   # path → refined_label_path

print('Generating refined pseudo-labels …')
with torch.no_grad():
    for batch in tqdm(full_loader, desc='Self-training inference'):
        imgs   = batch['image'].to(device)
        pseudo = batch['pseudo'].to(device)
        paths  = batch['path']

        with autocast('cuda', enabled=cfg.USE_AMP):
            v_pred, _ = model(imgs)

        # Ensemble: model (60%) + original Frangi (40%)
        refined = (0.60 * v_pred + 0.40 * pseudo).cpu().numpy()   # [B,1,H,W]

        for pred_arr, orig_path in zip(refined, paths):
            stem   = os.path.splitext(os.path.basename(orig_path))[0]
            rp     = os.path.join(REFINED_DIR, f'{stem}_refined.npy')
            np.save(rp, pred_arr[0].astype(np.float32))
            refined_map[orig_path] = rp

# ── Update dataframe with refined paths ──────────────────────────────────────
df_refined = df_all.copy()
df_refined['pseudo_path'] = df_refined['path'].map(
    lambda p: refined_map.get(p, df_all.loc[df_all['path'] == p, 'pseudo_path'].values[0])
)

tr2_df, vl2_df  = train_test_split(df_refined, test_size=0.15,
                                    random_state=42,
                                    stratify=df_refined['category'])
train_loader2   = DataLoader(RetinalDataset(tr2_df, cfg, augment=True),
                              batch_size=cfg.BATCH_SIZE, shuffle=True,
                              drop_last=True, **LOADER_KWARGS)
val_loader2     = DataLoader(RetinalDataset(vl2_df, cfg, augment=False),
                              batch_size=4, shuffle=False, **LOADER_KWARGS)

# ── Fine-tune at lower LR ─────────────────────────────────────────────────────
for g in optimizer.param_groups:
    g['lr'] = cfg.LR * 0.10

scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS_PHASE2, eta_min=1e-7)

print('=' * 60)
print('PHASE 2 — Self-training with refined labels')
print('=' * 60)

for epoch in range(cfg.NUM_EPOCHS_PHASE2):
    tr = run_epoch(model, train_loader2, criterion, optimizer, scaler,
                   device, cfg, train=True,
                   desc=f'P2 Epoch {epoch+1}/{cfg.NUM_EPOCHS_PHASE2} [train]')
    vl = run_epoch(model, val_loader2,   criterion, optimizer, scaler,
                   device, cfg, train=False,
                   desc=f'P2 Epoch {epoch+1}/{cfg.NUM_EPOCHS_PHASE2} [val]')
    scheduler2.step()

    history['train'].append(tr)
    history['val'].append(vl)

    tag = ''
    if vl['total'] < best_loss:
        best_loss = vl['total']
        torch.save({'epoch': cfg.NUM_EPOCHS_PHASE1 + epoch,
                    'model_state_dict': model.state_dict(),
                    'val_loss': best_loss}, CKPT_PATH)
        tag = '  ← best'

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Ep {epoch+1:03d}  '
              f'train={tr["total"]:.4f}  '
              f'val={vl["total"]:.4f}{tag}')

print(f'\nPhase 2 done. Best val loss: {best_loss:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 ─ Inference helper
# ─────────────────────────────────────────────────────────────────────────────
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def segment_image(img_path: str, model, cfg, device) -> dict:
    """
    Returns
    -------
    dict with keys: img_rgb, img_clahe, frangi_prob, frangi_bin,
                    vessel_map, vessel_bin, recon
    """
    model.eval()
    img_rgb, img_cl, g_cl = preprocess_retinal(img_path, cfg.IMG_SIZE)
    fp, fb                = make_pseudo_label(g_cl, cfg.FRANGI_SIGMAS)

    t = torch.from_numpy(img_cl.transpose(2, 0, 1)).float().div(255.0)
    inp = ((t.unsqueeze(0) - _MEAN) / _STD).to(device)

    with torch.no_grad():
        with autocast('cuda', enabled=cfg.USE_AMP):
            vp, rp = model(inp)

    vmap = vp.squeeze().cpu().numpy().astype(np.float32)
    reco = rp.squeeze().permute(1, 2, 0).cpu().numpy()

    # Post-processing
    thr    = threshold_otsu(vmap)
    vbin   = remove_small_objects((vmap > thr), min_size=20)
    vbin   = ndimage.binary_fill_holes(vbin).astype(np.uint8)

    return dict(img_rgb=img_rgb, img_clahe=img_cl,
                frangi_prob=fp, frangi_bin=fb.astype(np.uint8),
                vessel_map=vmap, vessel_bin=vbin, recon=reco)


# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH)['model_state_dict'])
model.eval()
print('Best model loaded for inference.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 ─ Per-category qualitative visualisation
# ─────────────────────────────────────────────────────────────────────────────

def show_result(res: dict, title: str = '', save_path: str = None):
    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    panels = [
        (res['img_rgb'],     'Original',              None),
        (res['img_clahe'],   'CLAHE Enhanced',         None),
        (res['img_clahe'][:,:,1], 'Green Channel',    'gray'),
        (res['recon'],       'Reconstructed',          None),
        (res['frangi_prob'], 'Frangi Vesselness',      'hot'),
        (res['frangi_bin'],  'Frangi Binary',          'gray'),
        (res['vessel_map'],  'Model Vessel Map ▶',    'hot'),
        (res['vessel_bin'],  'Model Segmentation ▶',  'gray'),
    ]
    for ax, (img, ttl, cmap) in zip(axes.flatten(), panels):
        ax.imshow(img, cmap=cmap)
        color = 'crimson' if '▶' in ttl else 'black'
        ax.set_title(ttl, fontsize=11, fontweight='bold', color=color)
        ax.axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    plt.close()


print('Visualising one sample per category …\n')
for cat in cfg.CATEGORIES:
    sub = df_all[df_all['category'] == cat]
    if len(sub) == 0:
        continue
    sample_path = sub.sample(1, random_state=1).iloc[0]['path']
    res = segment_image(sample_path, model, cfg, device)
    sp  = os.path.join(cfg.OUTPUT_DIR, f'seg_{cat}.png')
    show_result(res, title=cat.replace('_', ' '), save_path=sp)
    print(f'  Saved → {sp}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 ─ Vessel morphology metrics (no GT needed)
# ─────────────────────────────────────────────────────────────────────────────

def vessel_metrics(vbin: np.ndarray) -> dict:
    """Compute unsupervised morphology metrics from a binary vessel mask."""
    binary  = vbin.astype(bool)
    density = binary.mean()

    skel    = skeletonize(binary)
    skel_px = int(skel.sum())

    # Connected component stats
    lbl    = label(binary)
    props  = regionprops(lbl)
    n_seg  = len(props)

    if props:
        areas      = np.array([p.area          for p in props])
        perims     = np.array([p.perimeter      for p in props if p.perimeter > 0])
        avg_width  = float(np.mean(areas / (perims + 1e-6))) if len(perims) else 0.0
        avg_ecc    = float(np.mean([p.eccentricity for p in props]))
    else:
        avg_width, avg_ecc = 0.0, 0.0

    return {
        'vessel_density'  : float(density),
        'skeleton_px'     : skel_px,
        'n_vessel_segs'   : n_seg,
        'avg_vessel_width': avg_width,
        'avg_eccentricity': avg_ecc,
        'skel_density'    : float(skel.mean()),
    }


print('Running advanced analysis on all images …')
records = []

for idx, row in tqdm(df_all.iterrows(), total=len(df_all), desc='Analysis'):
    try:
        res  = segment_image(row['path'], model, cfg, device)
        mets = vessel_metrics(res['vessel_bin'])
        mets['category'] = row['category']
        mets['path']     = row['path']
        records.append(mets)
    except Exception as e:
        print(f'  [SKIP] {os.path.basename(row["path"])}: {e}')

analysis_df = pd.DataFrame(records)
CSV_PATH    = os.path.join(cfg.OUTPUT_DIR, 'vessel_analysis.csv')
analysis_df.to_csv(CSV_PATH, index=False)
print(f'\nAnalysis saved → {CSV_PATH}')
print(analysis_df.groupby('category').mean(numeric_only=True).round(4).to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 ─ Advanced Analysis Dashboard
# ─────────────────────────────────────────────────────────────────────────────

METRICS = [
    ('vessel_density',   'Vessel Density'),
    ('skeleton_px',      'Skeleton Length (px)'),
    ('avg_vessel_width', 'Avg Vessel Width'),
    ('n_vessel_segs',    'N Vessel Segments'),
    ('avg_eccentricity', 'Avg Eccentricity'),
]
CATS   = cfg.CATEGORIES
COLORS = sns.color_palette('husl', len(CATS))

fig  = plt.figure(figsize=(26, 20))
gs   = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.35)

# ── Box plots ────────────────────────────────────────────────────────────────
positions = [(0,0), (0,1), (0,2), (1,0), (1,1)]
for (r, c), (col, lbl) in zip(positions, METRICS):
    ax   = fig.add_subplot(gs[r, c])
    data = [analysis_df.loc[analysis_df['category'] == cat, col].dropna().values
            for cat in CATS]
    bp   = ax.boxplot(data, labels=[c[:9] for c in CATS],
                      patch_artist=True, notch=False, widths=0.55)
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set(facecolor=color, alpha=0.72)
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    ax.grid(axis='y', alpha=0.30)

# ── Dataset pie chart ─────────────────────────────────────────────────────────
ax_pie = fig.add_subplot(gs[1, 2])
cnts   = analysis_df['category'].value_counts()
ax_pie.pie(cnts.values, labels=cnts.index, autopct='%1.1f%%',
           colors=COLORS, startangle=90, textprops={'fontsize': 8})
ax_pie.set_title('Dataset Distribution', fontweight='bold', fontsize=10)

# ── Violin: vessel density per class ─────────────────────────────────────────
ax_vln = fig.add_subplot(gs[2, :2])
sns.violinplot(data=analysis_df, x='category', y='vessel_density',
               palette='husl', ax=ax_vln, inner='box', linewidth=1)
ax_vln.set_title('Vessel Density Distribution per Category',
                  fontweight='bold', fontsize=11)
ax_vln.set_xlabel('Category', fontsize=9)
ax_vln.set_ylabel('Vessel Density', fontsize=9)
ax_vln.tick_params(axis='x', rotation=30, labelsize=8)
ax_vln.grid(axis='y', alpha=0.25)

# ── Correlation heatmap ───────────────────────────────────────────────────────
ax_cor = fig.add_subplot(gs[2, 2])
corr   = analysis_df[[m for m, _ in METRICS]].corr()
im     = ax_cor.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ticks  = [m.replace('_', '\n') for m, _ in METRICS]
ax_cor.set_xticks(range(len(METRICS)))
ax_cor.set_yticks(range(len(METRICS)))
ax_cor.set_xticklabels(ticks, fontsize=7)
ax_cor.set_yticklabels(ticks, fontsize=7)
plt.colorbar(im, ax=ax_cor, fraction=0.046)
for i in range(len(METRICS)):
    for j in range(len(METRICS)):
        ax_cor.text(j, i, f'{corr.values[i,j]:.2f}',
                    ha='center', va='center', fontsize=7)
ax_cor.set_title('Metric Correlation', fontweight='bold', fontsize=10)

# ── Mean vessel density bar chart ────────────────────────────────────────────
ax_bar = fig.add_subplot(gs[3, :])
summary = analysis_df.groupby('category').agg(
    mean_density=('vessel_density', 'mean'),
    std_density =('vessel_density', 'std'),
    n           =('vessel_density', 'count')
).reset_index()
bars = ax_bar.bar(summary['category'], summary['mean_density'],
                   color=COLORS, alpha=0.80, edgecolor='black', linewidth=0.8)
ax_bar.errorbar(summary['category'], summary['mean_density'],
                 yerr=summary['std_density'], fmt='none', color='black',
                 capsize=5, linewidth=1.5)
for bar, (_, row2) in zip(bars, summary.iterrows()):
    ax_bar.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + row2['std_density'] + 0.002,
                f'{row2["mean_density"]:.4f}\n(n={row2["n"]})',
                ha='center', va='bottom', fontsize=8)
ax_bar.set_title('Mean Vessel Density per Category (± std)',
                  fontweight='bold', fontsize=11)
ax_bar.set_ylabel('Vessel Density', fontsize=9)
ax_bar.tick_params(axis='x', labelsize=9)
ax_bar.grid(axis='y', alpha=0.25)

fig.suptitle(
    'Retinal Vessel Segmentation — Advanced Analysis\n'
    'Self-Supervised Attention U-Net + ASPP  |  Fully Unsupervised',
    fontsize=14, fontweight='bold', y=1.01
)
ANALYSIS_FIG = os.path.join(cfg.OUTPUT_DIR, 'advanced_analysis.png')
plt.savefig(ANALYSIS_FIG, dpi=140, bbox_inches='tight')
plt.show()
print(f'Dashboard saved → {ANALYSIS_FIG}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 18 ─ Training loss curves
# ─────────────────────────────────────────────────────────────────────────────

def plot_history(history: dict, cfg) -> None:
    tr_tot = [h['total']  for h in history['train']]
    vl_tot = [h['total']  for h in history['val']]
    tr_ves = [h['vessel'] for h in history['train']]
    vl_ves = [h['vessel'] for h in history['val']]
    tr_rec = [h['recon']  for h in history['train']]
    vl_rec = [h['recon']  for h in history['val']]
    eps    = range(1, len(tr_tot) + 1)
    ph2    = cfg.NUM_EPOCHS_PHASE1

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for ax, (tr, vl, lbl) in zip(axes, [
        (tr_tot, vl_tot, 'Total Loss'),
        (tr_ves, vl_ves, 'Vessel Loss'),
        (tr_rec, vl_rec, 'Reconstruction Loss'),
    ]):
        ax.plot(eps, tr, 'steelblue',   lw=2, label='Train')
        ax.plot(eps, vl, 'tomato',      lw=2, label='Val')
        ax.axvline(ph2, color='gray', ls='--', lw=1.4,
                   label=f'Phase 2 start (ep {ph2})')
        ax.set_title(lbl, fontweight='bold', fontsize=12)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.30)

    plt.suptitle('Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    sp = os.path.join(cfg.OUTPUT_DIR, 'training_history.png')
    plt.savefig(sp, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Training history saved → {sp}')

plot_history(history, cfg)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 19 ─ Overlay grid: model segmentation on original images
# ─────────────────────────────────────────────────────────────────────────────

def build_overlay_grid(df, model, cfg, device, n_per_cat: int = 2):
    """Montage of (original | vessel overlay) for each category."""
    n_cats = len(cfg.CATEGORIES)
    fig, axes = plt.subplots(n_cats, n_per_cat * 2,
                              figsize=(n_per_cat * 8, n_cats * 4))

    for ri, cat in enumerate(cfg.CATEGORIES):
        sub     = df[df['category'] == cat]
        samples = sub.sample(min(n_per_cat, len(sub)),
                             random_state=7).iterrows()
        ci = 0
        for _, row2 in samples:
            res = segment_image(row2['path'], model, cfg, device)

            # Overlay: original + red vessel mask
            overlay = res['img_rgb'].copy()
            mask    = res['vessel_bin'].astype(bool)
            overlay[mask] = np.array([220, 20, 60], dtype=np.uint8)  # crimson

            axes[ri, ci * 2].imshow(res['img_rgb'])
            axes[ri, ci * 2].set_title(f'{cat[:12]} – orig', fontsize=8)
            axes[ri, ci * 2].axis('off')

            axes[ri, ci * 2 + 1].imshow(overlay)
            axes[ri, ci * 2 + 1].set_title(f'{cat[:12]} – overlay', fontsize=8,
                                            color='crimson')
            axes[ri, ci * 2 + 1].axis('off')
            ci += 1

    plt.suptitle('Vessel Segmentation Overlay Grid', fontsize=14,
                  fontweight='bold')
    plt.tight_layout()
    sp = os.path.join(cfg.OUTPUT_DIR, 'overlay_grid.png')
    plt.savefig(sp, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Overlay grid saved → {sp}')

build_overlay_grid(df_all, model, cfg, device, n_per_cat=2)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 20 ─ Final summary
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 65)
print('  UNSUPERVISED RETINAL VESSEL SEGMENTATION — COMPLETE')
print('=' * 65)
print(f'  Model   : Self-Supervised Attention U-Net + ASPP')
print(f'  Params  : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Dataset : {len(df_all)} images across {len(cfg.CATEGORIES)} categories')
print(f'  Training: Phase-1 ({cfg.NUM_EPOCHS_PHASE1} ep, Frangi labels)  +  '
      f'Phase-2 ({cfg.NUM_EPOCHS_PHASE2} ep, self-training)')
print()
print('  Per-category mean vessel density:')
print(analysis_df.groupby('category')['vessel_density']
      .mean().round(5).to_string())
print()
print('  Outputs saved to:', cfg.OUTPUT_DIR)
for fname in ['best_model.pth', 'vessel_analysis.csv',
              'advanced_analysis.png', 'training_history.png',
              'overlay_grid.png']:
    marker = '✓' if os.path.exists(os.path.join(cfg.OUTPUT_DIR, fname)) else '✗'
    print(f'    [{marker}]  {fname}')
print('=' * 65)